# Lab 3: Testing Forecasts Over Several Years, and a Planning Pack

**Time:** about 75 minutes

Labs 1 and 2 came to opposite conclusions. For contribution inflows, a simple seasonal baseline beat SARIMA (Seasonal
AutoRegressive Integrated Moving Average) and Prophet. For benefit payouts, XGBoost (eXtreme Gradient Boosting) beat
the baseline comfortably. Both conclusions rest on **one** test year. One year can be lucky or unlucky, just as one
exam doesn't fully show how good a student is.

In this lab you'll write a small piece of code that tests every model in exactly the same way, across **four**
years, and then use the winners to produce what finance actually needs: a forecast for next year with an honest
range around it.

### The idea: pretend to stand in the past

Imagine it's the end of December 2021. You only know the data up to that date. You forecast 2022, then wait a year and
see how you did. Then you move forward to December 2022, forecast 2023, and so on. Each of those "pretend today"
dates is called an **origin**.

```
                ┌── origin Dec 2021 ─► forecast 2022 ─► errors
series ──► for ─┼── origin Dec 2022 ─► forecast 2023 ─► errors  ──► average error per model ──► choose
                ├── origin Dec 2023 ─► forecast 2024 ─► errors      interval coverage
                └── origin Dec 2024 ─► forecast 2025 ─► errors
```

This is called **rolling-origin backtesting**. "Back" because you test on the past; "rolling origin" because the
pretend-today date rolls forward one year at a time. You never use data from after the origin, so every test is as
fair as a real forecast would have been.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **backtest** | testing a forecasting method on past periods, as if you'd used it at the time |
| **origin** | the "pretend today" date: the model may only use data up to here |
| **step** | how many months ahead a forecast is: step 1 = next month, step 12 = a year ahead |
| **interval / range** | low and high values around a forecast; an 80% interval should contain the actual value 8 times in 10 |
| **coverage** | the share of actual values that really fell inside the intervals |
| **percentile** | the value below which a given share of the data falls; the 10th percentile has 10% below it |
| **median** | the middle value: half the values are below it, half above |
| **simulation / sample path** | one plausible version of the future, drawn at random by the model; many of them show the spread |
| **planning pack** | the final tables and chart you'd hand to finance |

## Where this lab fits

**What you've done so far.** In Lab 1 you forecast contribution inflows and found the seasonal-naive baseline beat
SARIMA and Prophet on 2025. In Lab 2 you forecast benefit payouts and found XGBoost beat the baselines on 2025. On
Day 2 you met the same idea you'll use today: a backtest across several years, so one lucky year can't decide.

**What you'll do now.** You'll make all five forecasting methods look the same from the outside, test each one on both
series from four different "pretend today" dates, rank them on the evidence, put honest ranges on the methods that
don't give their own, and assemble a 2026 planning pack for finance: monthly forecasts with ranges, annual totals, and
the net surplus available to invest.

**Why it matters.** Finance and the investment committee will make real decisions from these numbers. A model chosen
on one year can be the wrong model, and a forecast without an honest range invites over-confident plans. This lab is
how a forecasting team earns the right to be believed.

**What happens next.** This completes Day 4. On Day 5 you move from planning cash to protecting it: finding unusual and
fraudulent benefit payments.

## Imports and settings

The same libraries as Labs 1 and 2. The `logging` and `warnings` lines come *before* the imports here, because some of
Prophet's messages appear while it's being imported. `time` is used to measure how long each model takes.

`ORIGINS` lists the four "pretend today" dates: the end of 2021, 2022, 2023 and 2024.

In [ ]:
import logging
import time
import warnings

# Hide library notices and Prophet's progress messages. (Before the imports, because some appear while importing.)
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("prophet.plot").disabled = True

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX      # SARIMA, from Lab 1
from xgboost import XGBRegressor                            # XGBoost, from Lab 2
from prophet import Prophet                                 # Prophet, from Lab 1

# statsmodels turns some warnings back on when it is imported, so switch them off again.
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (11, 4)
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREY = "#8a8984"

SEED = 42        # same random choices every run
HORIZON = 12     # forecast 12 months ahead

# The four "origins": the dates you pretend to be standing at when you make each forecast.
ORIGINS = ["2021-12-01", "2022-12-01", "2023-12-01", "2024-12-01"]

> **Where we are.** The libraries are loaded and the four origins (end of 2021, 2022, 2023 and 2024) are set. First,
> rebuild the two series from Labs 1 and 2.

## 1. The two series

The same two series as Labs 1 and 2, in TZS (Tanzanian shillings) billions:

- **inflows:** contribution cash received, by the month it arrived (Lab 1);
- **payouts:** total benefits paid per month (Lab 2).

The steps are the same as before, just shorter.

In [ ]:
# Inflows: cash received per payment month (as in Lab 1).
contrib = pd.read_csv("contributions_monthly_clean.csv", parse_dates=["payment_date"])
paid = contrib[contrib["payment_date"].notna()].copy()                       # only rows that were paid
paid["month"] = paid["payment_date"].dt.to_period("M").dt.to_timestamp()     # first day of the payment month

inflows = paid.groupby("month")["amount_paid_tzs"].sum() / 1e9               # TZS billions per month
inflows = inflows.loc["2017-03-01":"2025-12-01"].asfreq("MS")                # trim the incomplete edges

Load the payouts and put both series in a dictionary called `SERIES`. A dictionary lets you loop over "inflows" and
"payouts" by name later, instead of writing every step twice.

In [ ]:
# Payouts: total benefits paid per month (as in Lab 2).
pay = pd.read_csv("benefit_payouts_monthly.csv", parse_dates=["month"])
payouts = pay.groupby("month")["amount_paid_tzs"].sum().asfreq("MS") / 1e9

# Keep both in a dictionary, so you can loop over them.
SERIES = {"inflows": inflows, "payouts": payouts}

plt.plot(inflows.index, inflows.values, color=BLUE, label="contribution inflows")
plt.plot(payouts.index, payouts.values, color=ORANGE, label="benefit payouts")
plt.ylabel("TZS billions")
plt.legend()
plt.show()

> **Where we are.** Both series are in one dictionary, `SERIES`, so the code can loop over them by name. Next, the
> five models need to be made interchangeable.

### Theory: functions as building blocks

In Python a function is a value, like a number or a table. You can put it in a variable, store it in a dictionary, and
pass it to another function. That's what makes a fair test of many models easy:

```
MODELS = {"Naive": naive, "SARIMA": sarima, ...}     # names -> functions

for name in MODELS:
    model_function = MODELS[name]                    # pick a function
    forecast = model_function(train, 12)             # call it, like any other function
```

The testing code never needs to know *how* a model works, only that it takes the training series and a horizon and
returns a table with a `forecast` column. Agreeing on that shape is called defining an **interface**. It means that a
model somebody writes next year can be tested by the same code with no changes, which you'll do yourself in the last
exercise.

## 2. Make every model look the same from the outside

You want to test five different models with one piece of testing code. That only works if every model can be
**called in the same way and returns the same shape of answer**. So agree on a simple rule that every model follows:

> A model is a function that takes the training series and a horizon (how many months to forecast), and returns a
> table with one row per forecast month, a `forecast` column and, if it can, `lower` and `upper` columns for an 80%
> interval.

It's like agreeing that every report submitted to a committee uses the same template: the committee can then read any
report without learning a new format. Anything that follows the rule can be tested by the same code, including models
you haven't written yet.

Every model needs the list of months it's forecasting, so start with a small helper. Given the training data, it
returns the next `horizon` month-start dates.

In [ ]:
def future_months(train, horizon):
    """The `horizon` calendar months straight after the end of the training data."""
    first = train.index[-1] + pd.offsets.MonthBegin(1)          # the month after the last training month
    return pd.date_range(first, periods=horizon, freq="MS")    # that month and the ones after it


# Example: the 3 months after the end of 2024.
future_months(inflows[:"2024-12-01"], 3)

Now the five models, one cell each. Each is the code from Lab 1 or 2, wrapped in a function that follows the rule.
After each function there's a quick test: forecast 3 months from the end of 2024, so you can see what it returns.

**Model 1: naive.** Every future month equals the last month seen.

In [ ]:
def naive(train, horizon):
    """Baseline 1: every future month equals the last month seen."""
    months = future_months(train, horizon)
    return pd.DataFrame({"forecast": train.iloc[-1]}, index=months)      # the last value, repeated


naive(inflows[:"2024-12-01"], 3)

**Model 2: seasonal naive with growth.** The same month last year, times last year's growth (the Lab 1 winner).

In [ ]:
def seasonal_naive_growth(train, horizon):
    """Baseline 2: the same month last year, scaled up by last year's growth (horizon of up to 12 months)."""
    months = future_months(train, horizon)
    growth = train.iloc[-12:].sum() / train.iloc[-24:-12].sum()     # last 12 months vs the 12 before
    last_year = train.iloc[-12:].values[:horizon]                   # the same months, one year earlier
    return pd.DataFrame({"forecast": last_year * growth}, index=months)


seasonal_naive_growth(inflows[:"2024-12-01"], 3)

**Model 3: SARIMA.** The same model as Lab 1, on the log of the series, with an 80% interval, converted back to TZS
billions with `np.exp`.

In [ ]:
def sarima(train, horizon):
    """SARIMA on the log of the series, with an 80% interval, converted back to TZS billions."""
    months = future_months(train, horizon)
    model = SARIMAX(np.log(train), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    result = model.get_forecast(horizon)
    interval = np.exp(result.conf_int(alpha=0.2))          # 80% interval, back from logs
    return pd.DataFrame({
        "forecast": np.exp(result.predicted_mean).values,
        "lower": interval.iloc[:, 0].values,               # first column = low end
        "upper": interval.iloc[:, 1].values,               # second column = high end
    }, index=months)


sarima(inflows[:"2024-12-01"], 3)

**Model 4: Prophet.** The same set-up as Lab 1: yearly pattern, the COVID-19 flag for April to September 2020, an 80%
interval, all on logs.

In [ ]:
def prophet(train, horizon):
    """Prophet with yearly seasonality and a COVID-19 flag, with an 80% interval."""
    months = future_months(train, horizon)

    # Prophet's table: ds (date), y (log value), covid (1 for Apr-Sep 2020).
    history = pd.DataFrame({"ds": train.index, "y": np.log(train.values)})
    is_covid = (history["ds"] >= "2020-04-01") & (history["ds"] <= "2020-09-30")
    history["covid"] = is_covid.astype(int)

    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                    interval_width=0.8, changepoint_prior_scale=0.05)
    model.add_regressor("covid")
    model.fit(history)

    # Forecast the future months, with no COVID.
    future = pd.DataFrame({"ds": months, "covid": 0})
    prediction = model.predict(future)
    return pd.DataFrame({
        "forecast": np.exp(prediction["yhat"]).values,
        "lower": np.exp(prediction["yhat_lower"]).values,
        "upper": np.exp(prediction["yhat_upper"]).values,
    }, index=months)


prophet(inflows[:"2024-12-01"], 3)

**Model 5: XGBoost, recursive.** The Lab 2 approach in two functions:

- `lag_features` builds the feature table (lags, rolling averages of earlier months, calendar month);
- `xgboost_recursive` trains the model, then forecasts one month at a time, feeding each prediction back into the
  history so the next month's lags can be calculated.

In [ ]:
def lag_features(y):
    """Lags, rolling averages of earlier months, and the calendar month (the same idea as Lab 2)."""
    table = pd.DataFrame({"y": y})
    for lag in [1, 2, 3, 6, 12]:
        table["lag_" + str(lag)] = y.shift(lag)                                # the value `lag` months earlier
    previous = y.shift(1)                                                      # never include this month
    for window in [3, 6, 12]:
        table["rolling_mean_" + str(window)] = previous.rolling(window).mean()
    table["month_of_year"] = table.index.month
    return table


def xgboost_recursive(train, horizon):
    """XGBoost on lag features, forecasting one month at a time (as in Lab 2)."""
    # Train on the feature table.
    data = lag_features(train).dropna()
    model = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                         min_child_weight=3, random_state=SEED)
    model.fit(data.drop(columns="y"), data["y"])

    # Forecast month by month, feeding each prediction back into the history.
    history = train.copy()
    for month in future_months(train, horizon):
        history.loc[month] = np.nan                                  # an empty row for the new month
        row = lag_features(history).drop(columns="y").iloc[[-1]]     # its features
        history.loc[month] = float(model.predict(row)[0])            # predict, and keep it as if real

    return pd.DataFrame({"forecast": history.iloc[-horizon:].values}, index=future_months(train, horizon))


xgboost_recursive(inflows[:"2024-12-01"], 3)

Put all five functions in one dictionary, by name. Notice that the values are the **functions themselves** (no
brackets after the name): `MODELS["SARIMA"]` is the `sarima` function, and `MODELS["SARIMA"](train, 12)` calls it.

In [ ]:
# All five, by name. Because they follow the same rule, the testing code can treat them alike.
MODELS = {
    "Naive": naive,
    "Seasonal naive + growth": seasonal_naive_growth,
    "SARIMA": sarima,
    "Prophet": prophet,
    "XGBoost (recursive)": xgboost_recursive,
}

> **Where we are.** All five models follow the same rule and sit in one dictionary, `MODELS`. Each has been tried on a
> three-month forecast so you can see what it returns. Now the backtest itself.

### Theory: why four years tell you more than one

Each test year is one sample from a noisy world. 2025 might have been calm, or had an unusual June, or a payment
pattern that happened to suit one method. Averaging over several years cancels much of that luck out, in the same way
that a student's average over four exams says more than one exam.

Look out for two things in the results:

- **Consistent winners.** A method that ranks near the top in every year is more trustworthy than one that wins big
  once and loses the other three times.
- **The worst year.** Planning has to survive bad years, so the worst-year error is often as important as the average.

Every origin uses only data up to that date. If any information from after the origin crept in (for example by
choosing model settings using all the data), the backtest would flatter the models, which is the forecasting version of
Day 2's leakage.

## 3. The backtest

First do one backtest by hand, for one model (SARIMA) and one origin (December 2023), so you can see each step:

1. take the data up to the origin as training data;
2. take the 12 months after the origin as the "actual" values to check against;
3. forecast those 12 months from the training data alone;
4. put forecast and actual side by side.

In [ ]:
origin = "2023-12-01"

# 1. Train on everything up to the origin.
train = inflows[:origin]

# 2. The 12 months after the origin are the "actual" values to compare against.
actual = inflows[inflows.index > origin].iloc[:HORIZON]

# 3. Forecast them with the model.
forecast = sarima(train, HORIZON)

# 4. Put forecast and actual side by side.
forecast["actual"] = actual
forecast.head()

The function below repeats those four steps for every origin and stacks the results into one tall table. It adds two
columns that help later:

- `origin`: which pretend-today date the row came from;
- `step`: how far ahead the forecast was (1 = one month ahead ... 12 = a year ahead).

`reindex(actual.index)` lines the forecast rows up with the actual months, so each forecast sits next to the right
actual value.

In [ ]:
def evaluate_forecast(model_function, series, origins=ORIGINS, horizon=HORIZON):
    """Rolling-origin backtest: one row per origin and forecast month, with forecast and actual."""
    results = []
    for origin in origins:
        train = series[:origin]                                  # data up to the origin
        actual = series[series.index > origin].iloc[:horizon]    # the 12 months after it

        forecast = model_function(train, horizon)
        forecast = forecast.reindex(actual.index)        # line the forecast up with the actual months
        forecast["actual"] = actual
        forecast["origin"] = pd.Timestamp(origin)
        forecast["step"] = range(1, len(actual) + 1)     # 1 = one month ahead, 2 = two months ahead ...
        results.append(forecast)
    return pd.concat(results)                            # stack the four years into one table


# Try it on one model: 4 origins x 12 months = 48 rows.
bt = evaluate_forecast(seasonal_naive_growth, inflows)
bt.head(14)

Then turn a backtest table into a few numbers you can compare:

- **MAE, RMSE, MAPE**: the average errors over all 48 forecasts, as in Labs 1 and 2;
- **worst year MAPE**: the error of the single worst of the four years. An average can hide one bad year, and finance
  cares about how wrong you *can* be, not only how wrong you usually are;
- **80% interval coverage** (only for models that give an interval): the share of actual values that fell inside the
  interval. It should be close to 0.80.

In [ ]:
def summarise(bt):
    """Average errors, the worst year's MAPE, and the 80% interval coverage of a backtest."""
    error = bt["actual"] - bt["forecast"]
    percentage_error = error.abs() / bt["actual"]

    summary = {
        "MAE": round(float(error.abs().mean()), 3),
        "RMSE": round(float(np.sqrt((error ** 2).mean())), 3),
        "MAPE %": round(float(percentage_error.mean() * 100), 2),
        # The MAPE of each origin's year, then the largest of them.
        "worst year MAPE %": round(float(percentage_error.groupby(bt["origin"]).mean().max() * 100), 2),
    }
    # Only models with an interval have "lower" and "upper" columns.
    if "lower" in bt.columns:
        inside = (bt["actual"] >= bt["lower"]) & (bt["actual"] <= bt["upper"])   # True when inside the range
        summary["80% interval coverage"] = round(float(inside.mean()), 2)        # share of True
    return summary


summarise(bt)

Now run every model on both series: 2 series × 5 models × 4 origins = 40 forecasts. The two loops go through every
series and every model; for each pair, the backtest table is saved in `backtests` (you'll need it later) and its
summary becomes one row of the results. `time.perf_counter()` measures how long each took.

Prophet and SARIMA are fitted 4 times each per series, so give it a minute.

In [ ]:
backtests = {}      # (series name, model name) -> backtest table
rows = []           # one summary row per series and model

for series_name in SERIES:
    for model_name in MODELS:
        start = time.perf_counter()                                   # start the stopwatch
        bt = evaluate_forecast(MODELS[model_name], SERIES[series_name])
        backtests[(series_name, model_name)] = bt

        row = summarise(bt)
        row["series"] = series_name
        row["model"] = model_name
        row["seconds"] = time.perf_counter() - start                  # how long it took
        rows.append(row)
        print("done:", series_name, model_name)

# One table, indexed by series and model.
summary = pd.DataFrame(rows).set_index(["series", "model"])
summary.round(3)

Look at each series in turn, best (lowest MAE) first. `summary.loc["inflows"]` picks the inflows rows.

In [ ]:
print("inflows: ranked by MAE across four test years")
summary.loc["inflows"].sort_values("MAE")[["MAE", "MAPE %", "worst year MAPE %"]].round(2)

In [ ]:
print("payouts: ranked by MAE across four test years")
summary.loc["payouts"].sort_values("MAE")[["MAE", "MAPE %", "worst year MAPE %"]].round(2)

Now compare with what Labs 1 and 2 told you, because **both conclusions flip**:

- **Inflows.** In Lab 1, on 2025 alone, the seasonal naive baseline beat SARIMA and Prophet. Over four years, SARIMA
  is clearly best and the baseline is only third. 2025 happened to be a year that suited the baseline.
- **Payouts.** In Lab 2, on 2025 alone, XGBoost won comfortably. Over four years, the plain naive forecast is level
  with it or slightly ahead on MAE (mean absolute error). XGBoost's big win was largely a 2025 story.

If you'd shipped either Lab 1's or Lab 2's conclusion, you'd have shipped the wrong model. That's the whole case for
backtesting in one table.

Three more things to read off this:

1. **The ranking.** Does the Lab 1 and Lab 2 conclusion survive four years of testing, or was it a one-year fluke?
2. **The worst year.** An average hides the bad year. Finance cares about how wrong you can be, not only how wrong
   you usually are.
3. **Interval coverage.** An 80% interval should contain about 80% of actual values. Much lower means the model is
   overconfident. Much higher means the intervals are too wide to be useful.

**Your turn (1).** Error usually grows with the horizon. Using `backtests[("inflows", "SARIMA")]` (or any other
model), compute the mean absolute percentage error by `step` (1 to 12 months ahead) and plot it. How much worse is
month 12 than month 1?

In [ ]:
# Your code here

> **Where we are.** Over four years both of the earlier conclusions flipped: SARIMA is best for inflows, and the plain
> naive forecast is level with XGBoost for payouts. You also know each model's worst year and whether its intervals were
> honest. But three of the five models give no range at all.

### Theory: percentiles, and building a range from past mistakes

A **percentile** splits a list of values by share. For 100 numbers sorted from smallest to largest, the 10th percentile
is (roughly) the 10th number, and the 90th percentile is the 90th. 80% of the numbers lie between them.

Apply that to a model's past mistakes. Suppose that, three months ahead, a model's actual-divided-by-forecast ratios in
the backtest were:

```
0.94   0.97   0.99   1.01   1.02   1.03   1.06   1.09
```

The 10th percentile is about 0.96 and the 90th about 1.07. So for a new three-month-ahead forecast of 10.0 billion, an
80% range is about 9.6 to 10.7 billion: *"if the model is as wrong as it usually has been, the actual will most likely
land here"*.

This needs no theory about the model. It's simply the model's own track record, used honestly. Its weakness is that it
needs plenty of past mistakes to be reliable: with only four origins, each step has just four ratios, so treat these
bands as rough.

## 4. Ranges for models that don't produce them

The naive baselines and XGBoost give a single number, with no range. You can still put an honest range around them,
using the errors they **actually made** in the backtest.

The idea, step by step:

1. For every past forecast, divide the actual by the forecast. 1.05 means "the actual came in 5% above the forecast";
   0.97 means "3% below".
2. Group those ratios by step (1 month ahead, 2 months ahead ...), because errors are usually bigger further ahead.
3. For each step, find the 10th and 90th **percentiles** of the ratios: 10% of past errors were below the first, 10%
   above the second. So 80% of past errors fell between them.
4. Multiply the new forecast by those two ratios to get the low and high ends.

This is a simple form of **conformal prediction**. It makes no assumptions about how the model works, just "this is
how wrong it's been before, so expect about the same".

In [ ]:
def empirical_band(bt, forecast, low=0.10, high=0.90):
    """Add lower/upper columns to a forecast, from the model's own past errors in the backtest."""
    # For every past forecast: actual / forecast. 1.05 means the actual came in 5% above the forecast.
    ratio = bt["actual"] / bt["forecast"]

    # For each step ahead, the 10th and 90th percentile of those ratios.
    low_ratio = ratio.groupby(bt["step"]).quantile(low)
    high_ratio = ratio.groupby(bt["step"]).quantile(high)

    # Apply them to the new forecast, step by step.
    forecast = forecast.copy()
    steps = range(1, len(forecast) + 1)
    forecast["lower"] = forecast["forecast"].values * low_ratio.loc[steps].values
    forecast["upper"] = forecast["forecast"].values * high_ratio.loc[steps].values
    return forecast

When you look at the planning pack below, you may notice that some of the payout bands for the naive forecast don't
even contain the forecast line. That isn't a bug. Payouts keep growing, so "same as last month" is systematically too
low, and the band, built from real past errors, sits above it to correct for that. A biased forecast with a corrected
band is usable, but it's also a strong hint that a model which understands growth would serve you better. Which is
why you'd look at the second-ranked model before signing off.

With four origins there are only four errors per step, so these bands are rough. With more history you'd use more
origins, for example one every quarter. That's an easy change to `ORIGINS`.

> **Where we are.** Every model can now carry an 80% range: either its own, or one built from its backtest mistakes.
> You know which model to trust for each series. Time to build what finance asked for.

## 5. The planning pack

Now produce the forecast finance asked for: every month of 2026, for both series, each with an 80% range, plus the
annual totals.

Choose the model for each series from the backtest (by MAE here, but you could weigh the worst year too).
`idxmin()` returns the *name* of the row with the smallest value, which here is the model name.

In [ ]:
# idxmin() gives the name of the row with the smallest MAE.
choice = {}
choice["inflows"] = summary.loc["inflows"]["MAE"].idxmin()
choice["payouts"] = summary.loc["payouts"]["MAE"].idxmin()
choice

For each series, run the chosen model on **all** the data (up to December 2025) to forecast 2026. If that model has no
range of its own, add one with `empirical_band`, using its errors from the backtest. Then put the two forecasts side
by side in one table, one row per month.

In [ ]:
# Forecast 2026 with each chosen model, trained on all the data. Add a band if the model has none.
plan = {}
for series_name in SERIES:
    model_name = choice[series_name]
    forecast = MODELS[model_name](SERIES[series_name], HORIZON)       # call the chosen model
    if "lower" not in forecast.columns:
        forecast = empirical_band(backtests[(series_name, model_name)], forecast)
    plan[series_name] = forecast

# Side by side, one row per month.
pack = pd.concat([plan["inflows"][["forecast", "lower", "upper"]],
                  plan["payouts"][["forecast", "lower", "upper"]]],
                 axis=1, keys=["inflows", "payouts"])
pack.index = pack.index.strftime("%b %Y")         # "Jan 2026", "Feb 2026", ...
pack.round(2)

The chart for the pack: actual values since 2023 (solid lines), the 2026 forecasts (dashed), and the 80% ranges
(shaded). The vertical line marks the start of 2026.

In [ ]:
plt.figure(figsize=(11, 4.5))
for series_name, colour in [("inflows", BLUE), ("payouts", ORANGE)]:
    actual = SERIES[series_name].loc["2023":]
    forecast = plan[series_name]
    plt.plot(actual.index, actual.values, color=colour, lw=1.2, label=series_name + ", actual")
    plt.plot(forecast.index, forecast["forecast"], color=colour, lw=2, ls="--",
             label=series_name + ", forecast (" + choice[series_name] + ")")
    plt.fill_between(forecast.index, forecast["lower"], forecast["upper"], color=colour, alpha=0.15)   # the range

plt.axvline(pd.Timestamp("2026-01-01"), color=GREY, lw=0.8)
plt.title("2026 planning forecast with 80% ranges (TZS billions)", loc="left")
plt.legend(ncol=2, fontsize=9)
plt.show()

### Theory: simulation, and why totals need it

Each month's forecast is uncertain. When you add up twelve uncertain months, what's the uncertainty of the total? It's
**not** the sum of the twelve monthly ranges. That would only be right if every month came in at its low end together
(or at its high end together). In reality some months come in high and others low, and much of it cancels out.

The easiest honest way to get the range of a total is **simulation** (also called Monte Carlo, after the casino):

1. Ask the model for one random, plausible version of the whole year: twelve monthly values that vary the way real
   months do.
2. Add them up. That's one plausible annual total.
3. Repeat 500 times.
4. The middle of the 500 totals is your central estimate; the 10th and 90th percentiles give an 80% range.

The same trick works for any quantity worked out from uncertain inputs, such as **inflows minus payouts**: subtract
year by year across the simulations, then look at the spread of the results.

### An annual total, with an honest range

In Lab 1 you saw that adding up the monthly low ends exaggerates the annual range: it assumes every month comes in low
at the same time, which almost never happens.

The right way is to **simulate** whole years and look at the spread of their totals. Prophet can draw random
**sample paths**: each one is a plausible version of 2026 in which some months come in high and others low, as they
would in reality. Draw 500 of them, add up each one, and you have 500 plausible annual totals. Their spread is an
honest range for the year.

`predictive_samples` returns a table with one row per month and one column per simulated year; `sum(axis=0)` adds up
each column.

In [ ]:
def prophet_annual_samples(series, horizon=HORIZON, n_samples=500):
    """Simulate n_samples plausible versions of the coming year with Prophet, and return each one's total."""
    history = pd.DataFrame({"ds": series.index, "y": np.log(series.values)})
    is_covid = (history["ds"] >= "2020-04-01") & (history["ds"] <= "2020-09-30")
    history["covid"] = is_covid.astype(int)

    # uncertainty_samples: how many plausible years to simulate.
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                    changepoint_prior_scale=0.05, uncertainty_samples=n_samples)
    model.add_regressor("covid")
    model.fit(history)

    future = pd.DataFrame({"ds": future_months(series, horizon), "covid": 0})
    samples = np.exp(model.predictive_samples(future)["yhat"])     # one row per month, one column per simulated year
    return samples.sum(axis=0)                                      # the total of each simulated year


annual = {}
annual["inflows"] = prophet_annual_samples(inflows)
annual["payouts"] = prophet_annual_samples(payouts)
print(len(annual["inflows"]), "simulated 2026 totals for each series")

Summarise the 500 totals for each series: the **median** (the middle one, your central estimate) and the 10th and 90th
percentiles (an 80% range). The 2025 actual total is shown alongside for comparison.

In [ ]:
# The middle (median) and the 10th and 90th percentiles of the simulated totals.
rows = []
for series_name in ["inflows", "payouts"]:
    draws = annual[series_name]
    rows.append({
        "series": series_name,
        "2026 central": np.median(draws),
        "low (10%)": np.percentile(draws, 10),
        "high (90%)": np.percentile(draws, 90),
        "2025 actual": SERIES[series_name].loc["2025"].sum(),
    })
annual_table = pd.DataFrame(rows).set_index("series")
annual_table.round(1)

Finally, the number the investment team cares most about: **inflows minus payouts**, the new money left over to invest.
Subtract the simulated payouts from the simulated inflows, year by year, and summarise the result the same way.

In [ ]:
# Inflows minus payouts, simulated year by simulated year.
net = annual["inflows"] - annual["payouts"]
print(f"2026 net contribution surplus (inflows minus payouts): {np.median(net):.1f} bn TZS")
print(f"80% range {np.percentile(net, 10):.1f} to {np.percentile(net, 90):.1f}")

That net surplus is the number the investment team plans around: the new money available to invest. The honest
version of the message is:

> *"We expect about X billion in 2026, and we're 80% confident it will be between Y and Z. The main risk to the
> downside is a repeat of 2020-style payment disruption, which none of the models can foresee."*

Note that this annual range comes from Prophet even if another model won the monthly backtest. That's fine,
as long as you say so. Different questions can use different tools.

> **Where we are.** The planning pack is complete: monthly forecasts with ranges for both series, a chart, annual totals
> with honest ranges, and the net surplus, with a sentence a committee can act on.

## 6. Wrap-up

What you built today is a small version of the forecasting process a finance or actuarial team runs:

| Step | Today |
|---|---|
| Define the quantity carefully | Cash received vs contributions due |
| Always have a baseline | Naive and seasonal naive |
| Test honestly | Rolling-origin backtest over four years |
| Choose on evidence | The table in Section 3 |
| Quantify uncertainty | Model intervals, empirical bands, simulated annual totals |
| Communicate | The planning pack and one plain-language sentence |

On Day 10, one of the capstone options is to put this forecast behind an API (application programming interface: a way
for one program to call another) and monitor it as each month's actual figures arrive.

---

## Looking back, looking ahead

**What you did today.** Day 4 covered forecasting from start to finish:

- **Lab 1:** a monthly cash series, decomposed and forecast with baselines, SARIMA and Prophet.
- **Lab 2:** forecasting recast as supervised learning, with XGBoost on lag features and one global model.
- **Lab 3:** every method tested the same way over four years, honest ranges for every method, and a planning pack
  with simulated annual totals.

**The ideas to carry forward.**

- One test period can mislead; both of the earlier conclusions flipped over four years.
- Agree on a common interface and the same test code works for every model.
- A model's own past mistakes can give it an honest range.
- Ranges for totals come from simulating whole years, not from adding monthly bounds.
- Say plainly what the models can't foresee.

**Tomorrow (Day 5).** You'll move from forecasting cash to protecting it: finding unusual benefit payments with no
labels at all, learning from the few confirmed fraud cases, and building a review queue the investigations team can
work through.

## Your turn (bigger)

2. **Add a model.** Write a function `ets(train, horizon)` using `statsmodels.tsa.holtwinters.ExponentialSmoothing`
   with additive trend and multiplicative seasonality (`trend="add", seasonal="mul", seasonal_periods=12`). Add it to
   `MODELS` and rerun the backtest. Where does it rank?
3. **Quarterly origins.** Change `ORIGINS` to one every quarter from 2021 to 2024. Do the rankings or the interval
   coverage change? Why are more origins better?

In [ ]:
# Your code here

## Suggested answers

**(1)** Error by horizon. Calculate the percentage error of every forecast, then average it for each step ahead:

In [ ]:
bt = backtests[("inflows", "SARIMA")]

# Absolute percentage error of every forecast, then the average for each step ahead.
bt["ape"] = (bt["actual"] - bt["forecast"]).abs() / bt["actual"] * 100
by_step = bt.groupby("step")["ape"].mean()

plt.plot(by_step.index, by_step.values, marker="o", color=BLUE)
plt.title("SARIMA on inflows: error by months ahead")
plt.xlabel("months ahead")
plt.ylabel("MAPE %")
plt.show()

**(2)** Holt-Winters exponential smoothing. It forecasts with a weighted average of past values, giving recent months
more weight, plus a trend and a seasonal pattern. `seasonal="mul"` means the seasonal swing is a *percentage* of the
level, so it grows as the series grows. Because the function follows the same rule as the others, the existing
backtest code tests it unchanged:

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing


def ets(train, horizon):
    """Holt-Winters exponential smoothing: trend plus a seasonal pattern that scales with the level."""
    model = ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12).fit()
    return pd.DataFrame({"forecast": model.forecast(horizon).values}, index=future_months(train, horizon))


print("inflows:", summarise(evaluate_forecast(ets, inflows)))
print("payouts:", summarise(evaluate_forecast(ets, payouts)))